In [1]:
!pip -q install python-docx sentence-transformers faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

docx_path = next(iter(uploaded.keys()))
print("Loaded:", docx_path)

Saving molecular biology note .docx to molecular biology note .docx
Loaded: molecular biology note .docx


In [3]:
from docx import Document
import re

def read_docx_blocks(path):
    doc = Document(path)
    lines = []
    for p in doc.paragraphs:
        t = p.text.strip()
        if t:
            lines.append(t)
    return lines

def is_heading(line: str) -> bool:
    # Heuristic: ALL CAPS or ends with ":" or looks like a title line
    if len(line) <= 70 and line.isupper():
        return True
    if len(line) <= 70 and line.endswith(":"):
        return True
    # Example: "DNA REPLICATION", "MECHANISM OF DNA REPLICATION" etc.
    if len(line) <= 70 and re.match(r"^[A-Z0-9 ,()\-]+$", line) and sum(c.isalpha() for c in line) > 6:
        return True
    return False

def group_by_heading(lines):
    blocks = []
    current_heading = "UNTITLED"
    current_text = []

    for line in lines:
        if is_heading(line):
            if current_text:
                blocks.append({"heading": current_heading, "text": "\n".join(current_text).strip()})
            current_heading = line.strip(":").strip()
            current_text = []
        else:
            current_text.append(line)

    if current_text:
        blocks.append({"heading": current_heading, "text": "\n".join(current_text).strip()})

    return blocks

lines = read_docx_blocks(docx_path)
blocks = group_by_heading(lines)

print("Total heading-blocks:", len(blocks))
print("Example block heading:", blocks[0]["heading"])
print("Example block preview:\n", blocks[0]["text"][:400])

Total heading-blocks: 45
Example block heading: KALINGA UNIVERSITY
Example block preview:
 Department of Biochemistry
Course – M.Sc. Biochemistry								             Semester – 2nd
Subject – Molecular Biology
Subject Code – MSBC203
Notes – Unit 1


In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # fast baseline
# If you later want stronger retrieval, try:
# EMBED_MODEL = "BAAI/bge-base-en-v1.5"

embedder = SentenceTransformer(EMBED_MODEL)

# Create normalized embeddings (so cosine similarity == inner product)
emb = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
emb = np.array(emb, dtype=np.float32)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product on normalized vectors = cosine similarity
index.add(emb)

print("FAISS index ready. Vectors:", index.ntotal, "Dim:", dim)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index ready. Vectors: 57 Dim: 384


In [6]:
def retrieve(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype=np.float32)

    scores, ids = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(ids[0]):
        results.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "heading": metas[idx]["heading"],
            "chunk_id": metas[idx]["chunk_id"],
            "text": chunks[idx]
        })
    return results

# Quick test (uses topics that exist in your notes like DNA replication + enzymes) :contentReference[oaicite:2]{index=2}
results = retrieve("Explain initiation and elongation in DNA replication", k=5)
for r in results:
    print(f"\n#{r['rank']} | score={r['score']:.3f} | heading={r['heading']} | chunk={r['chunk_id']}")
    print(r["text"][:350], "...")


#1 | score=0.785 | heading=MECHANISM OF DNA REPLICATION | chunk=0
DNA replication is a semi-conservative process that involves the unwinding of the double helix and the synthesis of new DNA strands. The process is mediated by a complex interplay of enzymes, proteins, and other molecules. Initiation The replication process begins with the unwinding of the double helix at a specific region called the origin of repl ...

#2 | score=0.736 | heading=DNA REPLICATION | chunk=1
at a specific region called the origin of replication. This process is mediated by an enzyme called helicase, which breaks the hydrogen bonds between the two strands. The resulting single-stranded DNA is then stabilized by single-strand binding proteins. Elongation The elongation stage involves the synthesis of new DNA strands by an enzyme called D ...

#3 | score=0.710 | heading=DNA REPLICATION | chunk=0
DNA replication is the process by which a cell makes an exact copy of its DNA before cell division. This process is

In [7]:
def rag_answer_light(query, k=5):
    hits = retrieve(query, k=k)

    context = "\n\n".join(
        [f"[{h['heading']} | chunk {h['chunk_id']} | score {h['score']:.3f}]\n{h['text']}"
         for h in hits]
    )

    # Lightweight “answer” (no LLM): just returns curated context + a short heuristic summary
    summary_lines = [
        f"- From **{hits[0]['heading']}**: {hits[0]['text'][:180].strip()}...",
    ]
    if len(hits) > 1:
        summary_lines.append(f"- Also relevant: **{hits[1]['heading']}** (score {hits[1]['score']:.3f}).")

    return {
        "query": query,
        "top_evidence": hits,
        "summary": "\n".join(summary_lines),
        "context": context
    }

out = rag_answer_light("What is the role of helicase, primase, and ligase in DNA replication?", k=5)
print(out["summary"])
print("\n--- Evidence Context ---\n")
print(out["context"][:1500], "...")

- From **The replisome consists of several key components**: Helicase: Unwinds the DNA double helix, creating a replication fork. Primase: Synthesizes short RNA primers that provide a starting point for DNA polymerase. DNA polymerase: Synthe...
- Also relevant: **The replisome performs several key functions** (score 0.725).

--- Evidence Context ---

[The replisome consists of several key components | chunk 0 | score 0.800]
Helicase: Unwinds the DNA double helix, creating a replication fork. Primase: Synthesizes short RNA primers that provide a starting point for DNA polymerase. DNA polymerase: Synthesizes new DNA strands by adding nucleotides to the growing chain. Proofreading and editing enzymes: Correct errors in DNA synthesis, ensuring high fidelity. Single-strand binding proteins: Stabilize the single-stranded DNA template, preventing secondary structure formation. Topoisomerase: Relaxes the tension in the DNA molecule, allowing the replication fork to move forward. Function of Re